# CogniStream — Moondream2 Fine-tuning

Fine-tunes moondream2's text model on NVIDIA-distilled video captions.
Vision encoder is frozen (official recommendation).

**Steps:**
1. Upload `distillation_dataset.jsonl` and keyframe images
2. Train (2 epochs, ~5-10 min on Colab GPU)
3. Download the fine-tuned model
4. Convert to GGUF for Ollama

In [ ]:
# 1. Install dependencies
# Pin transformers to a version compatible with moondream 2024 revisions
!pip install -q 'transformers==4.44.2' torch pillow einops accelerate peft bitsandbytes

In [ ]:
# 2. Upload your data
# Option A: Upload from Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy your data folder (upload distillation_dataset.jsonl + frames to Drive first)
# !cp /content/drive/MyDrive/cognistream_finetune/distillation_dataset.jsonl /content/
# !cp -r /content/drive/MyDrive/cognistream_finetune/frames/ /content/frames/

# Option B: Upload directly via Colab file upload
from google.colab import files
print("Upload distillation_dataset.jsonl:")
uploaded = files.upload()

In [ ]:
# 3. Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# 4. Load dataset
import json
from pathlib import Path

DATASET_PATH = "distillation_dataset.jsonl"

items = []
with open(DATASET_PATH) as f:
    for line in f:
        items.append(json.loads(line))

print(f"Loaded {len(items)} training examples")

# Check how many have valid image paths
# (paths will be different on Colab — we only use the text for training)
print(f"Sample response: {items[0]['response'][:200]}")

In [ ]:
# 5. Load moondream2
# Patch the config download to fix the rope_scaling issue before AutoModel parses it
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import hf_hub_download
import json as _json

MD_REVISION = "2024-08-26"
DEVICE = "cuda"
DTYPE = torch.float16

print(f"Loading moondream2 (revision={MD_REVISION})...")

# Download config first and patch it if needed
try:
    config = AutoConfig.from_pretrained(
        "vikhyatk/moondream2", revision=MD_REVISION, trust_remote_code=True,
    )
    # Patch broken rope_scaling
    if hasattr(config, "phi_config") and hasattr(config.phi_config, "rope_scaling"):
        rs = config.phi_config.rope_scaling
        if rs and "factor" not in rs:
            rs["factor"] = 1.0
            print("Patched rope_scaling.factor")
    if hasattr(config, "text_config") and hasattr(config.text_config, "rope_scaling"):
        rs = config.text_config.rope_scaling
        if rs and "factor" not in rs:
            rs["factor"] = 1.0
            print("Patched text_config.rope_scaling.factor")
except Exception as e:
    print(f"Config patch failed (may not be needed): {e}")
    config = None

tokenizer = AutoTokenizer.from_pretrained(
    "vikhyatk/moondream2", revision=MD_REVISION, trust_remote_code=True,
)

# Try loading with patched config first, fall back to plain load
try:
    model = AutoModelForCausalLM.from_pretrained(
        "vikhyatk/moondream2", revision=MD_REVISION, trust_remote_code=True,
        config=config, torch_dtype=DTYPE, device_map={"": DEVICE},
    )
except Exception as e:
    print(f"First load failed: {e}")
    print("Retrying without config override...")
    model = AutoModelForCausalLM.from_pretrained(
        "vikhyatk/moondream2", revision=MD_REVISION, trust_remote_code=True,
        torch_dtype=DTYPE, device_map={"": DEVICE},
    )
print("Model loaded.")

In [ ]:
# 6. Apply LoRA to text_model (memory-efficient fine-tuning)
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, TaskType

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Apply LoRA only to the text_model's Phi transformer
# Target the attention projections (Wqkv, out_proj) and MLP (fc1, fc2)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["Wqkv", "out_proj", "fc1", "fc2"],  # Phi-2 layer names
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to text_model only
text_model = get_peft_model(model.text_model, lora_config)
model.text_model = text_model

# Count parameters
trainable = sum(p.numel() for p in text_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in text_model.parameters())
print(f"LoRA trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.2f}%)")

# Verify text_model still works as PhiForCausalLM
print(f"\nText model: {type(text_model).__name__}")
print(f"Base: {type(text_model.base_model.model).__name__}")

In [ ]:
# 7. Training with LoRA (memory-efficient)
import random
import time

EPOCHS = 3
GRAD_ACCUM = 8
LR = 2e-4  # LoRA typically uses higher LR than full fine-tuning

optimizer = torch.optim.AdamW(
    [p for p in text_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)

text_model.train()
total_steps = (len(items) * EPOCHS) // GRAD_ACCUM
step = 0
t_start = time.monotonic()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

for epoch in range(EPOCHS):
    random.shuffle(items)
    epoch_loss = 0.0
    batch_loss = 0.0
    valid = 0
    nan_count = 0

    for i, item in enumerate(items):
        try:
            answer = item["response"]

            tokens = tokenizer(
                answer, return_tensors="pt",
                truncation=True, max_length=256, padding=False,
            )
            input_ids = tokens["input_ids"].to(DEVICE)

            if input_ids.size(1) < 5:
                continue

            outputs = text_model(input_ids=input_ids, labels=input_ids)
            loss = outputs.loss

            if not torch.isfinite(loss):
                nan_count += 1
                optimizer.zero_grad()
                # Clear cache to prevent fragmentation
                torch.cuda.empty_cache()
                continue

            loss = loss / GRAD_ACCUM
            loss.backward()
            batch_loss += loss.item()
            valid += 1

            if valid % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in text_model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                optimizer.zero_grad()
                step += 1

                if step % 5 == 0:
                    elapsed = time.monotonic() - t_start
                    mem = torch.cuda.memory_allocated() / 1e9
                    print(f"  Epoch {epoch+1} | Step {step}/{total_steps} | Loss: {batch_loss:.4f} | VRAM: {mem:.1f}GB | {elapsed/60:.1f} min")
                epoch_loss += batch_loss
                batch_loss = 0.0

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if i < 5:
                print(f"  Sample {i}: OOM — cleared cache and skipping")
            continue
        except Exception as exc:
            if i < 5:
                print(f"  Sample {i} error: {type(exc).__name__}: {exc}")
            continue

    n_steps = max(1, valid // GRAD_ACCUM)
    print(f"Epoch {epoch+1} complete | Avg loss: {epoch_loss/n_steps:.4f} | Valid: {valid}/{len(items)} | NaN: {nan_count}")

elapsed = time.monotonic() - t_start
print(f"\nTraining complete: {step} steps in {elapsed/60:.1f} min")

In [ ]:
# 8. Save LoRA adapter (small, ~50 MB)
OUTPUT_DIR = "/content/cognistream-moondream-lora"

# Save only the LoRA adapter weights (not the full base model)
text_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to: {OUTPUT_DIR}")

import os
total = sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fns in os.walk(OUTPUT_DIR) for f in fns)
print(f"Adapter size: {total/1e6:.1f} MB")

# Also save merged model if you want a standalone fine-tuned version
# (merges LoRA weights back into the base model — produces ~3 GB file)
MERGE = False  # Set True if you want the merged model

if MERGE:
    print("\nMerging LoRA into base model...")
    merged = text_model.merge_and_unload()
    MERGED_DIR = "/content/cognistream-moondream-merged"
    merged.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    total_m = sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fns in os.walk(MERGED_DIR) for f in fns)
    print(f"Merged model: {total_m/1e9:.1f} GB")

In [ ]:
# 9. Quick sanity test — generate a few tokens from the fine-tuned model
text_model.eval()

test_prompt = """Analyze this image. Respond with EXACTLY this format:
SCENE: [2-3 sentence description of setting and atmosphere]
OBJECTS: [comma-separated list of visible objects]
ACTIVITY: [one sentence describing what is happening]
ANOMALY: [anything unusual, or 'none']
SCENE: """

tokens = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    generated = text_model.generate(
        input_ids=tokens.input_ids,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

output = tokenizer.decode(generated[0], skip_special_tokens=True)
print("Generated continuation:")
print(output[len(test_prompt):])
print("\nModel is producing text. Format compliance will be better with image context.")

In [ ]:
# 10. Download the LoRA adapter (small, ~50 MB)
!cd /content && zip -r cognistream-moondream-lora.zip cognistream-moondream-lora/
files.download("/content/cognistream-moondream-lora.zip")

# Optional: save to Google Drive instead
# !cp -r /content/cognistream-moondream-lora /content/drive/MyDrive/cognistream_finetune/

In [ ]:
## After Download

1. **Extract** `cognistream-moondream-lora.zip` to `models/cognistream-moondream-lora/` in your project
2. **Load in your code**:
   ```python
   from transformers import AutoModelForCausalLM, AutoTokenizer
   from peft import PeftModel

   base = AutoModelForCausalLM.from_pretrained(
       "vikhyatk/moondream2", revision="2024-08-26",
       trust_remote_code=True, torch_dtype=torch.float16,
   )
   base.text_model = PeftModel.from_pretrained(
       base.text_model, "models/cognistream-moondream-lora"
   )
   ```
3. **Or merge the LoRA into the base** (produces standalone model):
   - Re-run cell 8 with `MERGE = True`
   - Download the larger merged model
   - Convert to GGUF with `scripts/finetune/export_ollama.py`
   - Use with Ollama as `cognistream-vlm`

**Note on Ollama integration**: Because moondream's GGUF format in Ollama uses a different architecture than the HuggingFace version, merging the LoRA and converting back to GGUF requires extra steps (see the moondream repo for the official conversion script). For direct Python inference the LoRA adapter works as-is.

In [ ]:
# 11. Benchmark: Base moondream vs LoRA fine-tuned (side-by-side)
# Uses moondream's native caption() method — requires images
# Download a few test frames if you don't have any uploaded

import os
import urllib.request

# Download test images if not already present
test_images = []
sample_urls = [
    ("traffic.jpg", "https://images.unsplash.com/photo-1519608487953-e999c86e7455?w=640"),
    ("kitchen.jpg", "https://images.unsplash.com/photo-1556909114-f6e7ad7d3136?w=640"),
    ("classroom.jpg", "https://images.unsplash.com/photo-1580582932707-520aed937b7b?w=640"),
    ("street.jpg", "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=640"),
    ("park.jpg", "https://images.unsplash.com/photo-1519681393784-d120267933ba?w=640"),
]

for name, url in sample_urls:
    path = f"/content/{name}"
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print(f"Failed to download {name}: {e}")
            continue
    test_images.append(path)

print(f"Test images: {len(test_images)}")

# Load BASE moondream (without LoRA) for comparison
from transformers import AutoModelForCausalLM as AMF

print("\nLoading BASE moondream (for comparison)...")
base_model = AMF.from_pretrained(
    "vikhyatk/moondream2", revision=MD_REVISION,
    trust_remote_code=True, torch_dtype=torch.float16,
    device_map={"": DEVICE},
)
base_model.eval()

# Our fine-tuned model is still in memory (model variable)
model.eval()

# Compare outputs
from PIL import Image

prompt = "Describe this image with scene, objects, activity, and anomaly."

print("\n" + "=" * 60)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 60)

for i, img_path in enumerate(test_images):
    img = Image.open(img_path).convert("RGB")
    print(f"\n--- Image {i+1}: {os.path.basename(img_path)} ---")

    # BASE moondream
    with torch.no_grad():
        base_embeds = base_model.encode_image(img)
        base_answer = base_model.answer_question(
            base_embeds, prompt, tokenizer,
        )
    print(f"BASE:     {base_answer[:300]}")

    # LoRA fine-tuned
    with torch.no_grad():
        ft_embeds = model.encode_image(img)
        ft_answer = model.answer_question(
            ft_embeds, prompt, tokenizer,
        )
    print(f"LoRA FT:  {ft_answer[:300]}")

# Save comparison as JSON for later review
import json
comparison = []
for img_path in test_images:
    img = Image.open(img_path).convert("RGB")
    with torch.no_grad():
        base_embeds = base_model.encode_image(img)
        base_ans = base_model.answer_question(base_embeds, prompt, tokenizer)
        ft_embeds = model.encode_image(img)
        ft_ans = model.answer_question(ft_embeds, prompt, tokenizer)
    comparison.append({
        "image": os.path.basename(img_path),
        "base": base_ans,
        "finetuned": ft_ans,
    })

with open("/content/benchmark_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)
print("\nComparison saved to /content/benchmark_comparison.json")

## After Download

1. Extract `cognistream-moondream.zip` to `data/finetune/cognistream-moondream/`
2. Convert to GGUF:
   ```bash
   python scripts/finetune/export_ollama.py
   ```
3. Set in `.env`:
   ```
   OLLAMA_MODEL=cognistream-vlm
   ```